# Unlimited OCR Drive–Colab worker

In Colab choose **Runtime → Change runtime type → GPU**, then run every cell. This worker is temporary. It stores original uploads and results in `MyDrive/UnlimitedOCR/`; the bearer token is intentionally not written to Drive.

In [ ]:
!pip -q install "transformers==4.57.1" accelerate fastapi uvicorn python-multipart pymupdf einops addict easydict psutil
print('Runtime pinned to transformers 4.57.1')

In [ ]:
# Install the model runtime. Colab download/install time depends on the selected GPU image.
!wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared.deb >/dev/null
from google.colab import drive
drive.mount('/content/drive')
import os, torch
assert torch.cuda.is_available(), 'Enable a GPU runtime, then run this notebook again.'
MODEL_DTYPE = torch.float16 if torch.cuda.get_device_capability(0)[0] < 8 else torch.bfloat16
from transformers import AutoModel, AutoTokenizer
MODEL_NAME = 'baidu/Unlimited-OCR'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, use_safetensors=True, torch_dtype=MODEL_DTYPE).eval().cuda().to(MODEL_DTYPE)
print('Model loaded on:', torch.cuda.get_device_name(0), 'using', MODEL_DTYPE)

In [ ]:
# Start the authenticated Drive-backed worker. Re-run this cell after a Colab disconnect.
import asyncio, json, os, queue, re, secrets, shutil, subprocess, threading, time, uuid, uvicorn
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import fitz
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware

ROOT = Path('/content/drive/MyDrive/UnlimitedOCR')
for name in ('input', 'jobs', 'output', 'logs', 'runtime'): (ROOT / name).mkdir(parents=True, exist_ok=True)
TOKEN = secrets.token_urlsafe(32)
MAX_BATCH_BYTES = 1_000_000_000
jobs, work_queue = {}, queue.Queue()
app = FastAPI(title='Unlimited OCR Colab Worker')
# Local file pages send Origin: null. The bearer token is mandatory on every route.
app.add_middleware(CORSMiddleware, allow_origins=['null'], allow_credentials=False, allow_methods=['GET','POST','OPTIONS'], allow_headers=['Authorization','Content-Type'])

def now(): return datetime.now(timezone.utc).isoformat()
def save_job(job):
    (ROOT / 'jobs' / f"{job['job_id']}.json").write_text(json.dumps(job, indent=2), encoding='utf-8')
def require_token(authorization):
    if not authorization or not secrets.compare_digest(authorization.removeprefix('Bearer ').strip(), TOKEN):
        raise HTTPException(401, 'Invalid bearer token')
def clean_relative(value):
    path = PurePosixPath(value.replace('\\', '/'))
    if path.is_absolute() or '..' in path.parts or not path.name: raise ValueError('Invalid relative path')
    return path
def pdf_pages(path, work):
    work.mkdir(parents=True, exist_ok=True); document = fitz.open(path); paths = []; matrix = fitz.Matrix(300/72, 300/72)
    for index, page in enumerate(document):
        output = work / f'page_{index + 1:04d}.png'; page.get_pixmap(matrix=matrix).save(output); paths.append(str(output))
    document.close(); return paths
def text_from_artifacts(output):
    values=[]
    for path in output.rglob('*'):
        if path.name in {'result.json','result.md','result.txt'} or not path.is_file(): continue
        if path.suffix.lower() in {'.txt','.md'}:
            try: values.append(path.read_text(encoding='utf-8'))
            except UnicodeDecodeError: pass
    return '\n\n'.join(values).strip()
def process(job):
    job['status']='running'; job['message']='Running GPU document parsing'; job['started_at']=now(); save_job(job)
    out = ROOT / 'output' / job['job_id']; out.mkdir(parents=True, exist_ok=True)
    try:
        for index, source in enumerate(job['files'], 1):
            input_path = ROOT / source['drive_path']; raw = out / f"item_{index:04d}"; raw.mkdir(exist_ok=True)
            if input_path.suffix.lower() == '.pdf':
                images = pdf_pages(input_path, raw / 'pages')
                with torch.inference_mode(), torch.amp.autocast('cuda', dtype=MODEL_DTYPE):
                    model.infer_multi(tokenizer, prompt='<image>Multi page parsing.', image_files=images, output_path=str(raw), image_size=1024, max_length=32768, no_repeat_ngram_size=35, ngram_window=1024, save_results=True)
            else:
                with torch.inference_mode(), torch.amp.autocast('cuda', dtype=MODEL_DTYPE):
                    model.infer(tokenizer, prompt='<image>document parsing.', image_file=str(input_path), output_path=str(raw), base_size=1024, image_size=640, crop_mode=True, max_length=32768, no_repeat_ngram_size=35, ngram_window=128, save_results=True)
            job['message'] = f'Processed {index}/{len(job["files"])} files'; save_job(job)
        extracted = text_from_artifacts(out)
        result = {'job_id': job['job_id'], 'status':'succeeded', 'created_at':job['created_at'], 'completed_at':now(), 'input_files':job['files'], 'drive_output_path':str(out), 'text':extracted}
        (out/'result.json').write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
        (out/'result.md').write_text('# OCR result\n\n' + (extracted or '_Model artifacts were saved; no text file was produced._'), encoding='utf-8')
        (out/'result.txt').write_text(extracted, encoding='utf-8')
        job.update(status='succeeded', message='Complete', completed_at=result['completed_at'], drive_output_path=str(out)); save_job(job)
    except Exception as exc:
        job.update(status='failed', message=str(exc), completed_at=now()); save_job(job)
def worker():
    while True:
        job = work_queue.get(); process(job); work_queue.task_done()
threading.Thread(target=worker, daemon=True).start()
@app.get('/health')
def health(authorization: str | None = Header(default=None)):
    require_token(authorization); return {'status':'ready','model':MODEL_NAME,'queue_depth':work_queue.qsize()}
@app.post('/jobs')
async def create_job(files: list[UploadFile] = File(...), relative_paths: str = Form(...), authorization: str | None = Header(default=None)):
    require_token(authorization)
    try: paths=json.loads(relative_paths)
    except json.JSONDecodeError: raise HTTPException(400, 'relative_paths must be JSON')
    if len(files) != len(paths) or not files: raise HTTPException(400, 'Files and relative paths do not match')
    job_id=uuid.uuid4().hex; input_root=ROOT/'input'/job_id; input_root.mkdir(parents=True); stored=[]; total=0
    try:
        for upload, relative in zip(files, paths):
            safe=clean_relative(relative); suffix=safe.suffix.lower()
            if suffix not in {'.png','.jpg','.jpeg','.pdf'}: raise HTTPException(415, f'Unsupported file: {safe.name}')
            target=input_root/safe; target.parent.mkdir(parents=True, exist_ok=True); written=0
            with target.open('wb') as handle:
                while chunk := await upload.read(1024*1024):
                    written += len(chunk); total += len(chunk)
                    if total > MAX_BATCH_BYTES: raise HTTPException(413, 'Batch exceeds 1 GB limit')
                    handle.write(chunk)
            stored.append({'name':safe.name,'relative_path':str(safe),'bytes':written,'drive_path':str(target.relative_to(ROOT))})
    except Exception:
        shutil.rmtree(input_root, ignore_errors=True); raise
    job={'job_id':job_id,'status':'queued','message':'Waiting for GPU worker','created_at':now(),'files':stored}; jobs[job_id]=job; save_job(job); work_queue.put(job); return {'job_id':job_id,'status':'queued'}
@app.get('/jobs/{job_id}')
def get_job(job_id: str, authorization: str | None = Header(default=None)):
    require_token(authorization); job=jobs.get(job_id)
    if not job: raise HTTPException(404, 'Unknown job')
    return job
@app.get('/jobs/{job_id}/result')
def get_result(job_id: str, authorization: str | None = Header(default=None)):
    require_token(authorization); output=ROOT/'output'/job_id/'result.json'
    if not output.exists(): raise HTTPException(409, 'Result is not ready')
    result=json.loads(output.read_text(encoding='utf-8')); result['result_url']=f"{PUBLIC_URL}/jobs/{job_id}/result"; return result

threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'), daemon=True).start()
tunnel = subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None
for _ in range(90):
    line=tunnel.stdout.readline()
    match=re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if match: PUBLIC_URL=match.group(0); break
if not PUBLIC_URL: raise RuntimeError('Cloudflare quick tunnel did not start; run this cell again.')
runtime={'run_id':uuid.uuid4().hex,'endpoint':PUBLIC_URL,'started_at':now(),'note':'Bearer token is intentionally not stored in Drive.'}
(ROOT/'runtime'/'endpoint.json').write_text(json.dumps(runtime, indent=2), encoding='utf-8')
print('\nWorker ready')
print('Endpoint:', PUBLIC_URL)
print('Bearer token (do not save in Drive):', TOKEN)